In [0]:
# Permite probar el manejo de fallos del Job.
dbutils.widgets.dropdown(
    "simular_fallo",
    "false",
    ["false", "true"]
)

if dbutils.widgets.get("simular_fallo") == "true":
    raise RuntimeError(
        "PRUEBA CONTROLADA: fallo simulado antes de revisar archivos."
    )

In [0]:
import csv
import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F


RUTA_FUENTE = "/Volumes/workspace/taco_bueno/archivos_fuente/"
TABLA_INVENTARIO = "workspace.default.audit_source_files"

# Los tres encabezados que identificamos en nuestra base.
columnas_base = [
    "Store #", "Date", "Total", "Check Number", "Time Card",
    "Begin Time", "End Time", "Total Discount", "Order Voided",
    "Tax", "Quantity", "Transaction Type", "Menu Item",
    "Menu Item Ext Ref", "Pop", "POS Code", "Menu Price",
    "Item Discount", "Item Voided"
]

columnas_21 = columnas_base + ["Report Group", "Tender Type"]
columnas_22 = columnas_21 + ["Total Rounding"]

encabezados_permitidos = [
    columnas_base,
    columnas_21,
    columnas_22
]

archivos = [
    archivo
    for archivo in dbutils.fs.ls(RUTA_FUENTE)
    if archivo.name.lower().endswith(".csv")
]

if not archivos:
    raise ValueError("No se encontraron archivos CSV en la carpeta fuente.")

registros = []

for archivo in archivos:
    encabezado = []
    estado = "Error"
    error = None
    esquema_valido = False

    try:
        ruta_local = archivo.path.removeprefix("dbfs:")

        with open(
            ruta_local,
            encoding="utf-8-sig",
            newline=""
        ) as contenido:
            lector = csv.reader(contenido)
            encabezado = next(lector, [])

            primera_fila = next(
                (
                    fila
                    for fila in lector
                    if any(valor.strip() for valor in fila)
                ),
                None
            )

        if not encabezado:
            estado = "Vacío"
        elif primera_fila is None:
            estado = "Solo encabezado"
        else:
            estado = "Con datos"

        esquema_valido = encabezado in encabezados_permitidos

    except Exception as excepcion:
        error = f"{type(excepcion).__name__}: {excepcion}"

    registros.append((
        archivo.name,
        archivo.path,
        archivo.size,
        archivo.modificationTime,
        len(encabezado),
        estado,
        esquema_valido,
        encabezado,
        error
    ))

inventario = spark.createDataFrame(
    registros,
    schema="""
        archivo STRING,
        ruta STRING,
        tamano_bytes LONG,
        modificado_ms LONG,
        columnas INT,
        estado STRING,
        esquema_valido BOOLEAN,
        encabezado ARRAY<STRING>,
        error STRING
    """
)

ejecucion_id = str(uuid.uuid4())
fecha_revision = datetime.now(timezone.utc)

inventario = (
    inventario
    .withColumn("ejecucion_id", F.lit(ejecucion_id))
    .withColumn("revisado_at", F.lit(fecha_revision))
)

# Cada ejecución agrega una revisión nueva al historial.
inventario.write.format("delta").mode("append").saveAsTable(
    TABLA_INVENTARIO
)

display(
    inventario
    .groupBy("estado", "columnas", "esquema_valido")
    .count()
    .orderBy("estado", "columnas")
)

print(f"Archivos revisados: {len(registros)}")
print(f"Identificador de revisión: {ejecucion_id}")

problemas = [
    registro[0]
    for registro in registros
    if not registro[6] or registro[8] is not None
]

if problemas:
    raise RuntimeError(
        "Revisar archivos con errores o encabezados no admitidos: "
        + ", ".join(problemas)
    )

print("Inventario guardado: todos los encabezados son admitidos.")